<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports](#imports)
- [Notebook Notes](#notebook-notes)
- [Load Selected Features](#load-selected-features)
- [Load Feature-Engineered Dataset](#load-feature-engineered-dataset)
- [Prepare Modeling Data](#prepare-modeling-data)
- [Train Random Forest With GridSearchCV](#train-random-forest-with-gridsearchcv)
- [Save Model and Metrics](#save-model-and-metrics)
- [Inspect Saved Metrics](#inspect-saved-metrics)
- [Optional Reproduce Model From JSON](#optional-reproduce-model-from-json)
- [Run Full Workflow](#run-full-workflow)


</div>

##### **IMPORTANT**:
- YOU MUST HAVE THE FEATURE-ENGINEERED DATASET `../Datasets/highest_snr_feature_engineered.parquet`
- YOU MUST HAVE THE SELECTED FEATURES JSON `../Reports/highest_snr_selected_features.json`
- The selected features JSON should be created by `highest_snr_feature_engineering_explore.py`

---

##### **ORIENTATION**:
- This notebook trains a Random Forest on selected engineered features, not flattened raw I/Q
- The trained model and reproducibility JSON are saved similarly to `highest_snr_baseline.py`

---

##### **SUMMARY**:
- The RFC model trained on the top 25 information gain features had a ROC of 97.3% compared to the RFC model trained on all engineered features at 97.8%
- Though the model dropped ~2-3% in accuracy, precision, recall, and ~0.5% in ROC, this indicates that training a model on the top 25 features as opposed to all 175 features engineered preserved much of the performance
- Additionally, during feature engineering, features were treated as **UNIVARIATE** which may lead to data poisoning or obfuscate some information gain when features are combined
- Future improvements likely will depend on exploring **MULTIVARIATE** information gain as opposed to a straightforward **UNIVARIATE** application

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports

- [Back to Table of Contents](#table-of-contents)

</div>

In [1]:
# Import helper module for feature-engineered modeling
import highest_snr_feature_engineering_modeling as m

# Helpful when actively editing highest_snr_feature_engineering_modeling.py
# Basically, I do not have to reload the entire notebook every time
from importlib import reload
m = reload(m)


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Notebook Notes

- [Back to Table of Contents](#table-of-contents)

</div>

This notebook is the modeling step that follows feature-engineering exploration.

The expected project flow is:

```text
1. highest_snr_feature_engineering.py
   - Create engineered features from the reduced highest-SNR .hdf5 dataset

2. highest_snr_feature_engineering_explore.py / .ipynb
   - Rank engineered features by information gain / mutual information
   - Save top feature names to ../Reports/highest_snr_selected_features.json

3. highest_snr_feature_engineering_modeling.py / .ipynb
   - Load selected top information-gain features
   - Train a RandomForestClassifier with GridSearchCV
   - Save the best model and metrics JSON
```


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Load Selected Features

- [Back to Table of Contents](#table-of-contents)

</div>

In [2]:
# Load the selected top information-gain features from the explore step
selected_feature_names, selected_features_payload = m.load_selected_feature_names(
    selected_features_filepath=m.SELECTED_FEATURES_INPUT_PATH
)

selected_features_payload


Selected feature names loaded successfully!
Selected features filepath: ../Reports/highest_snr_selected_features.json
Selection method: information_gain
Top N: 25
Number of selected features: 25



{'selection_method': 'information_gain',
 'top_n': 25,
 'selected_feature_names': ['magnitude_cv',
  'radius_std',
  'std_magnitude',
  'power_cv',
  'std_power',
  'iqr_magnitude',
  'max_power',
  'max_magnitude',
  'radius_unique_rounded_count',
  'q75_magnitude',
  'q25_magnitude',
  'std_inst_freq',
  'range_magnitude',
  'radius_mean',
  'mean_magnitude',
  'kurtosis_magnitude',
  'range_inst_freq',
  'iqr_inst_freq',
  'quadrant_balance_std',
  'kurtosis_inst_freq',
  'peak_to_average_power_ratio',
  'std_phase',
  'max_fft_magnitude',
  'max_fft_power',
  'std_fft_power']}

In [3]:
# View selected features in order of information-gain ranking
for idx, feature in enumerate(selected_feature_names, start=1):
    print(f"{idx}. {feature}")


1. magnitude_cv
2. radius_std
3. std_magnitude
4. power_cv
5. std_power
6. iqr_magnitude
7. max_power
8. max_magnitude
9. radius_unique_rounded_count
10. q75_magnitude
11. q25_magnitude
12. std_inst_freq
13. range_magnitude
14. radius_mean
15. mean_magnitude
16. kurtosis_magnitude
17. range_inst_freq
18. iqr_inst_freq
19. quadrant_balance_std
20. kurtosis_inst_freq
21. peak_to_average_power_ratio
22. std_phase
23. max_fft_magnitude
24. max_fft_power
25. std_fft_power


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Load Feature-Engineered Dataset

- [Back to Table of Contents](#table-of-contents)

</div>

In [4]:
# Load the feature-engineered tabular dataset
df_features = m.load_feature_engineered_dataset(
    feature_dataset_filepath=m.FEATURE_ENGINEERED_DATASET_PARQUET
)

df_features.head()


Feature-engineered dataset loaded successfully!
Dataset filepath: ../Datasets/highest_snr_feature_engineered.parquet
Dataset shape: (24000, 109)



,signal_index,modulation_id,modulation_type,snr,mean_I,std_I,min_I,max_I,median_I,range_I,...,quadrant_2_ratio,quadrant_3_ratio,quadrant_4_ratio,quadrant_balance_std,near_origin_ratio,far_origin_ratio,radius_mean,radius_std,radius_unique_rounded_count,phase_unique_rounded_count
0,0,0,OOK,30,0.739839,0.781117,-0.500421,2.351720,0.559564,2.852141,...,0.002930,0.228516,0.000000,0.313410,0.567383,0.230469,0.990590,0.845569,241,54
1,1,0,OOK,30,1.011886,1.002867,-0.734198,2.919468,0.918309,3.653667,...,0.179688,0.029297,0.640625,0.232467,0.535156,0.234375,1.110384,0.893427,242,28
2,2,0,OOK,30,0.811902,0.713866,-0.462071,2.154327,0.836580,2.616398,...,0.000000,0.179688,0.000000,0.337343,0.503906,0.240234,1.231032,0.885757,237,56
3,3,0,OOK,30,0.974050,0.837252,-0.564327,2.455967,1.050703,3.020294,...,0.173828,0.000000,0.826172,0.340138,0.500000,0.241211,1.225372,0.882576,241,58
4,4,0,OOK,30,0.626439,0.585162,-0.317435,1.814335,0.595447,2.131770,...,0.192383,0.000000,0.807617,0.331382,0.520508,0.243164,1.147916,0.892107,250,63


In [5]:
# Confirm dataset shape and target distribution
print("Dataset shape:", df_features.shape)
print()
print("Modulation distribution:")
print(df_features["modulation_type"].value_counts().sort_index())
print()
print("SNR distribution:")
print(df_features["snr"].value_counts().sort_index())


Dataset shape: (24000, 109)

Modulation distribution:
modulation_type
128APSK      1000
128QAM       1000
16APSK       1000
16PSK        1000
16QAM        1000
256QAM       1000
32APSK       1000
32PSK        1000
32QAM        1000
4ASK         1000
64APSK       1000
64QAM        1000
8ASK         1000
8PSK         1000
AM-DSB-SC    1000
AM-DSB-WC    1000
AM-SSB-SC    1000
AM-SSB-WC    1000
BPSK         1000
FM           1000
GMSK         1000
OOK          1000
OQPSK        1000
QPSK         1000
Name: count, dtype: int64

SNR distribution:
snr
30    24000
Name: count, dtype: int64


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Prepare Modeling Data

- [Back to Table of Contents](#table-of-contents)

</div>

In [6]:
# Prepare X/y using only the selected top information-gain features
X_features, y_labels, metadata_df, feature_columns_used = m.prepare_modeling_data(
    df_features=df_features,
    selected_feature_names=selected_feature_names,
    target_column="modulation_id"
)

print("X_features shape:", X_features.shape)
print("y_labels shape:", y_labels.shape)
print("metadata_df shape:", metadata_df.shape)


MODELING DATASET PREPARED
X_features shape: (24000, 25)
y_labels shape: (24000,)
metadata_df shape: (24000, 4)
Target column: modulation_id
Number of features used: 25

X_features shape: (24000, 25)
y_labels shape: (24000,)
metadata_df shape: (24000, 4)


In [7]:
# Quick sanity check: model features should match selected features
print("Selected feature count:", len(selected_feature_names))
print("Model feature count:", len(feature_columns_used))
print("Same feature list:", selected_feature_names == feature_columns_used)


Selected feature count: 25
Model feature count: 25
Same feature list: True


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Train Random Forest With GridSearchCV

- [Back to Table of Contents](#table-of-contents)

</div>

This step trains the final engineered-feature Random Forest model using the selected features from the exploration step.

The scoring metric matches the baseline style:

```python
scoring='roc_auc_ovr_weighted'
```

This can take a while depending on your machine.


In [8]:
best_model, metrics, cv_results_df = m.train_and_test_model(
    feature_dataset_filepath=m.FEATURE_ENGINEERED_DATASET_PARQUET,
    selected_features_filepath=m.SELECTED_FEATURES_INPUT_PATH,
    mod_type_mapping=m.MOD_TYPE_MAPPING,
    target_column="modulation_id",
    test_size=0.20,
    random_state=35,
    cv=3,
    rfc_n_jobs=1,
    grid_search_n_jobs=-1,
    use_selected_features=True,
    save_metrics_json=True,
    metrics_output_path=m.FEATURE_ENGINEERED_METRICS_OUTPUT_PATH,
    cv_results_output_path=m.FEATURE_ENGINEERED_CV_RESULTS_OUTPUT_PATH
)


Feature-engineered dataset loaded successfully!
Dataset filepath: ../Datasets/highest_snr_feature_engineered.parquet
Dataset shape: (24000, 109)

Selected feature names loaded successfully!
Selected features filepath: ../Reports/highest_snr_selected_features.json
Selection method: information_gain
Top N: 25
Number of selected features: 25

MODELING DATASET PREPARED
X_features shape: (24000, 25)
y_labels shape: (24000,)
metadata_df shape: (24000, 4)
Target column: modulation_id
Number of features used: 25

TRAIN/TEST SPLIT
Train rows: 19200
Test rows: 4800
X_train shape: (19200, 25)
X_test shape: (4800, 25)
y_train shape: (19200,)
y_test shape: (4800,)

ENGINEERED-FEATURE GRID SEARCH STARTED
Scoring metric: roc_auc_ovr_weighted
Total parameter combinations: 32
Features used: 25

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  12.4s
[CV] END max_depth=None, max

In [9]:
# Inspect the top GridSearchCV results
cv_results_df[[
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "std_train_score",
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split",
    "param_min_samples_leaf",
    "param_max_features"
]].sort_values("rank_test_score").head(10)


,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,param_n_estimators,param_max_depth,param_min_samples_split,param_min_samples_leaf,param_max_features
21,1,0.970932,0.000563,0.999992,1.177946e-06,200,30,2,2,sqrt
5,2,0.970913,0.000622,0.999992,1.089027e-06,200,None,2,2,sqrt
31,3,0.970804,0.000573,0.999983,1.914339e-06,200,30,5,2,log2
15,4,0.970754,0.000467,0.999982,1.515620e-06,200,None,5,2,log2
23,5,0.970702,0.000581,0.999984,1.957427e-06,200,30,5,2,sqrt
7,6,0.970699,0.000683,0.999984,1.899948e-06,200,None,5,2,sqrt
3,7,0.970686,0.000743,0.999998,3.790523e-07,200,None,5,1,sqrt
19,8,0.970678,0.000808,0.999998,4.227667e-07,200,30,5,1,sqrt
13,9,0.970485,0.000678,0.999992,1.202926e-06,200,None,2,2,log2
27,10,0.970471,0.000557,0.999997,6.608314e-07,200,30,5,1,log2


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Save Model and Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

In [10]:
# Save the best model and the reproducibility JSON
m.save_feature_engineered_model(
    best_model=best_model,
    metrics=metrics,
    model_output_path=m.FEATURE_ENGINEERED_MODEL_OUTPUT_PATH,
    metrics_output_path=m.FEATURE_ENGINEERED_METRICS_OUTPUT_PATH,
    save_model=True,
    save_metrics_json=True
)


FEATURE-ENGINEERED MODEL SAVE COMPLETE
Model saved to: ../Models/highest_snr_feature_engineered_random_forest_gridsearch.joblib
Metrics JSON saved to: ../Models/highest_snr_feature_engineered_random_forest_gridsearch_metrics.json



<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Inspect Saved Metrics

- [Back to Table of Contents](#table-of-contents)

</div>

In [11]:
# Inspect saved metrics dictionary in memory
print("Project stage:", metrics["project_stage"])
print("Input type:", metrics["input_type"])
print("Number of features used:", metrics["features_used"]["num_features_used"])
print("Best params:", metrics["grid_search"]["best_params"])
print("Test metrics:", metrics["test_metrics"])


Project stage: highest_snr_feature_engineering_modeling
Input type: selected_engineered_features
Number of features used: 25
Best params: {'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Test metrics: {'accuracy': 0.69375, 'precision_weighted': 0.6863657143056958, 'recall_weighted': 0.69375, 'roc_auc_ovr_weighted': 0.9729046648550725}


In [12]:
# Show top feature importances from the final tuned model
import pandas as pd

feature_importance_df = pd.DataFrame(metrics["feature_importances"])
feature_importance_df.head(25)


,feature,random_forest_importance,random_forest_importance_rank
0,std_inst_freq,0.073169,1
1,iqr_magnitude,0.061106,2
2,iqr_inst_freq,0.052672,3
3,std_power,0.052603,4
4,power_cv,0.051169,5
5,q75_magnitude,0.044935,6
6,max_magnitude,0.044518,7
7,magnitude_cv,0.044201,8
8,std_fft_power,0.043737,9
9,q25_magnitude,0.043732,10


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Optional Reproduce Model From JSON

- [Back to Table of Contents](#table-of-contents)

</div>

This optional step verifies that the model can be reproduced from the saved JSON recipe. It uses the saved dataset path, selected features, train/test indices, random state, and best hyperparameters.


In [13]:
# Optional: reproduce the model from the saved JSON
reproduced_model, reproduction_metrics = m.reproduce_model_from_json(
    metrics_json_filepath=m.FEATURE_ENGINEERED_METRICS_OUTPUT_PATH,
    save_model=False
)


Feature-engineered dataset loaded successfully!
Dataset filepath: ../Datasets/highest_snr_feature_engineered.parquet
Dataset shape: (24000, 109)

MODELING DATASET PREPARED
X_features shape: (24000, 25)
y_labels shape: (24000,)
metadata_df shape: (24000, 4)
Target column: modulation_id
Number of features used: 25

REPRODUCING FEATURE-ENGINEERED RANDOM FOREST MODEL
Metrics JSON filepath: ../Models/highest_snr_feature_engineered_random_forest_gridsearch_metrics.json
Feature dataset filepath: ../Datasets/highest_snr_feature_engineered.parquet
Train rows: 19200
Test rows: 4800
Features used: 25
Random state: 35
Best hyperparameters:
  max_depth: 30
  max_features: sqrt
  min_samples_leaf: 2
  min_samples_split: 2
  n_estimators: 200

REPRODUCED MODEL PERFORMANCE
Accuracy:              0.6937
Weighted Precision:    0.6864
Weighted Recall:       0.6937
Weighted ROC-AUC OVR:  0.9729



<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Run Full Workflow

- [Back to Table of Contents](#table-of-contents)

</div>

Instead of running each cell manually, you can run the whole workflow from the module.

This is commented out so the notebook does not accidentally retrain the model twice.


In [ ]:
# Optional: run the entire modeling workflow from the .py file
# m.main()
